# BT4012 — Bitcoin transaction classification

This standalone notebook contains the complete current tabular workflow: data
checks, rolling temporal validation, model comparison, final refitting, and CSV
submission generation. Run all cells from a fresh kernel using Python 3.12 and
`requirements.txt`. Place the four competition CSVs in `data/raw/` first.
No other notebook, script, report, or experiment file is required.

Class **0 = negative / licit**, class **1 = positive / illicit**. ROC-AUC is the
competition metric. Only labelled rows fit models; no graph features, pseudo-labels,
random primary split, extensive tuning, or automatic Kaggle submission are used.

The original Logistic Regression (E01) had local AUC **0.917141** and user-reported
public Kaggle AUC **0.88776**. Its unchanged specification is reevaluated here as
E01-R. Four boosted configurations (E02-E05) are compared under the original fixed
selection rule. E04 was selected in the recorded run; its public score is unknown.

Model settings, raw-file fingerprints, and historical experiment identifiers are
embedded below. This notebook reproduces the existing experiments without further
tuning. Before final coursework submission, rename it to your NUS student number,
for example `e1234567.ipynb`.


## 1. Data loading and configuration

Load transaction features and the submission template. Verify the four raw-file
SHA-256 fingerprints against constants embedded below; the edge list is checked
for integrity but never used in modelling. `txId` identifies transactions; the
literal test `index` column identifies submission rows. Missing labels remain
unknown and are excluded from fitting.

Tree models receive unscaled features with native NaN handling. Infinities become
NaN in working copies only. The Logistic Regression pipeline fits its median
imputer and standard scaler within each fitting fold. Feature meanings and their
original construction history are undocumented; no extra graph features are made.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import platform
import warnings
from importlib.metadata import version

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/raw/train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside the repository with the four CSVs in data/raw/.")
RAW = ROOT / "data/raw"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUT / ".matplotlib"))

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

SEED = 42
FEATURES = [f"feat_{i}" for i in range(1, 166)]
FOLDS = [(1, 20, 21, 25), (2, 25, 26, 30), (3, 30, 31, 35)]
LATEST_TOLERANCE = 0.002
MEAN_TOLERANCE = 0.001
VERSIONS = {name: version(name) for name in ["numpy", "pandas", "scikit-learn", "lightgbm", "xgboost", "threadpoolctl"]}
np.random.seed(SEED)

# One prespecified configuration per model family; only time inclusion varies.
LGB_PARAMS = dict(n_estimators=400, learning_rate=0.05, num_leaves=31,
                  max_depth=5, min_child_samples=40, subsample=0.8, subsample_freq=1,
                  colsample_bytree=1.0, reg_lambda=1.0, objective="binary",
                  random_state=SEED, n_jobs=1, verbosity=-1, deterministic=True,
                  force_col_wise=True, data_random_seed=SEED, bagging_seed=SEED,
                  feature_fraction_seed=SEED)
XGB_PARAMS = dict(n_estimators=400, learning_rate=0.05, max_depth=5,
                  min_child_weight=5, subsample=0.8, colsample_bytree=1.0,
                  reg_lambda=1.0, objective="binary:logistic", eval_metric="auc",
                  tree_method="hist", device="cpu", random_state=SEED, n_jobs=1)
LR_PARAMS = dict(C=1.0, penalty="l2", solver="lbfgs", class_weight="balanced",
                 max_iter=3000, tol=1e-4, random_state=SEED)

def fingerprint(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

# Raw-file fingerprints are embedded so no separate manifest is required.
EXPECTED_RAW_HASHES = {'sample_submission.csv': '3dba74ad6151053aa21f366fb9faf6da7393969a0ba6f8a2ebcec561871f739e', 'test.csv': '344e47c8cd8815f4072d7f6ecbecee8919ab648294a8eb4ed032a026c3eb91db', 'train.csv': '6f7e56883f0aa27147262a0d71954ad64702ca57f16fbf9ed58fae21a66dd706', 'txs_edgelist.csv': 'a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02dff084971b1b72a5156e'}
raw_hashes = {name: fingerprint(RAW / name) for name in EXPECTED_RAW_HASHES}
assert raw_hashes == EXPECTED_RAW_HASHES, "Raw inputs changed; inspect before comparing experiments."

# Historical E01 results; these are not reevaluated public leaderboard scores.
E01_LOCAL_AUC = 0.917141264532569
E01_PUBLIC_AUC = 0.88776  # Reported by the user after manual submission.
EXPECTED_CONFIG_IDS = {'E01-R': '093c95aee6480953caceba717728225668d37f234761689c526b19720b882d9b', 'E02': '51effe8c2f48587757ad99b54d9f23ef19b1c63518dfb4c8f1c41c7fe8a25e9b', 'E03': '71af9481844d36f7f3ccc9b299ddf9e449dbbe4954b97e2e7e7f396156255d94', 'E04': '739641a353d656a696e592db04371e7dc83a2ee0a760c61ea6e0123d76d30298', 'E05': '10f9f16294e01c7ae642f2c97c6e79405c5608befc069474efa547fd5d57587d'}

train = pd.read_csv(RAW / "train.csv", low_memory=False)
test = pd.read_csv(RAW / "test.csv", low_memory=False)
sample = pd.read_csv(RAW / "sample_submission.csv")
assert train.columns.tolist() == ["txId", "time_step", *FEATURES, "label"]
assert test.columns.tolist() == ["index", "txId", "time_step", *FEATURES]
assert sample.columns.tolist() == ["index", "target"]
assert set(train["label"].dropna().unique()) == {0, 1}
assert train["time_step"].notna().all() and train["time_step"].between(1, 35).all()
assert test["time_step"].notna().all() and test["time_step"].between(36, 49).all()
assert train["txId"].is_unique and test["txId"].is_unique
assert train["txId"].notna().all() and test["txId"].notna().all()
assert set(train["txId"]).isdisjoint(test["txId"])
for frame in [test, sample]:
    assert frame["index"].is_unique and frame["index"].notna().all()
assert len(test) == len(sample) and set(test["index"]) == set(sample["index"])
for frame in [train, test]:
    assert all(pd.api.types.is_numeric_dtype(frame[c]) for c in ["time_step", *FEATURES])

labelled = train.loc[train["label"].notna()].copy()
labelled["label"] = labelled["label"].astype(int)
X = labelled[["time_step", *FEATURES]].replace([np.inf, -np.inf], np.nan)
y = labelled["label"]
X_test = test[["time_step", *FEATURES]].replace([np.inf, -np.inf], np.nan)
quality = pd.DataFrame([
    {"dataset": name, "rows": len(frame), "missing_features": int(frame[FEATURES].isna().sum().sum()),
     "infinite_features": int(np.isinf(frame[FEATURES].to_numpy()).sum())}
    for name, frame in [("train", train), ("test", test)]
])
display(quality)
print("Labelled:", len(labelled), "| unlabelled excluded:", int(train['label'].isna().sum()))
print("Python:", platform.python_version(), "| packages:", VERSIONS)
print("Raw SHA-256 checks passed for all four files.")

,dataset,rows,missing_features,infinite_features
0,train,141772,0,0
1,test,15329,0,0


Labelled: 31235 | unlabelled excluded: 110537
Python: 3.12.4 | packages: {'numpy': '1.26.4', 'pandas': '2.2.2', 'scikit-learn': '1.5.1', 'lightgbm': '4.6.0', 'xgboost': '3.0.5', 'threadpoolctl': '3.5.0'}
Raw SHA-256 checks passed for all four files.


## 2. Temporal folds and unchanged Logistic Regression reference

Use expanding fitting windows and disjoint five-step validation windows. Every
model uses the same masks and row ordering. Both labels must occur in each side.
Imbalance weights use **fitting labels only**. The validation folds are never
passed for early stopping; all tree fits use a fixed 400 rounds.

E01-R reevaluates the original Logistic Regression specification on these folds
without changing E01 or creating another Logistic Regression submission. Mean AUC
is unweighted; standard deviation uses `ddof=0`. With only three folds and shared
fitting history, this spread is descriptive, not a confidence interval.

In [2]:
fold_counts = []
for fold, fit_end, val_start, val_end in FOLDS:
    fit = labelled["time_step"] <= fit_end
    val = labelled["time_step"].between(val_start, val_end)
    assert not (fit & val).any() and y[fit].nunique() == y[val].nunique() == 2
    assert labelled.loc[fit, "time_step"].max() < labelled.loc[val, "time_step"].min()
    fold_counts.append({"fold": fold, "fit_end": fit_end, "validation_start": val_start,
                        "validation_end": val_end, "fit_rows": int(fit.sum()),
                        "fit_0": int(y[fit].eq(0).sum()), "fit_1": int(y[fit].eq(1).sum()),
                        "validation_rows": int(val.sum()), "validation_0": int(y[val].eq(0).sum()),
                        "validation_1": int(y[val].eq(1).sum()),
                        "scale_pos_weight": float(y[fit].eq(0).sum() / y[fit].eq(1).sum())})
fold_counts = pd.DataFrame(fold_counts)
display(fold_counts)

fold_results = []

def evaluate_configuration(experiment_id, model_name, include_time, build_model):
    columns = ["time_step", *FEATURES] if include_time else FEATURES
    for fold, fit_end, val_start, val_end in FOLDS:
        fit = labelled["time_step"] <= fit_end
        val = labelled["time_step"].between(val_start, val_end)
        X_fit, X_val = X.loc[fit, columns], X.loc[val, columns]
        y_fit, y_val = y.loc[fit], y.loc[val]
        if X_fit.isna().all().any():
            raise ValueError("All-missing fitting feature; inspect before continuing.")
        model = build_model(y_fit)
        with warnings.catch_warnings(), threadpool_limits(limits=1):
            warnings.simplefilter("error", ConvergenceWarning)
            model.fit(X_fit, y_fit)
            positive = int(np.flatnonzero(model.classes_ == 1)[0])
            probabilities = model.predict_proba(X_val)[:, positive]
        assert np.isfinite(probabilities).all() and np.all((probabilities >= 0) & (probabilities <= 1))
        auc = float(roc_auc_score(y_val, probabilities))
        fold_results.append({"experiment_id": experiment_id, "model": model_name,
                             "include_time_step": include_time, "n_features": len(columns),
                             "fold": fold, "roc_auc": auc})
        print(f"{experiment_id} | {model_name} | time={include_time} | fold {fold} AUC={auc:.6f}", flush=True)


def build_lr(y_fit):
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scaler", StandardScaler()),
                     ("model", LogisticRegression(**LR_PARAMS))])

evaluate_configuration("E01-R", "Logistic Regression", False, build_lr)
assert abs(fold_results[-1]["roc_auc"] - 0.917141264532569) < 1e-9, "Unchanged E01 specification did not reproduce its latest fold."

,fold,fit_end,validation_start,validation_end,fit_rows,fit_0,fit_1,validation_rows,validation_0,validation_1,scale_pos_weight
0,1,20,21,25,18889,17118,1771,5311,4745,566,9.665726
1,2,25,26,30,24200,21863,2337,2705,2088,617,9.355156
2,3,30,31,35,26905,23951,2954,4330,3640,690,8.107989


E01-R | Logistic Regression | time=False | fold 1 AUC=0.894508
E01-R | Logistic Regression | time=False | fold 2 AUC=0.933691
E01-R | Logistic Regression | time=False | fold 3 AUC=0.917141


## 3. LightGBM experiments

E02 uses 165 features; E03 adds numeric `time_step`. The model learns nonlinear
splits and interactions with limited-depth trees. Use the fitting negative/positive
count ratio as `scale_pos_weight`; do not combine it with another class-weighting
mechanism. No global scaling/imputation is required; the estimator handles NaNs.

CPU determinism and column-wise histogram construction are fixed explicitly.
These settings follow the [LightGBM 4.6 documentation](https://lightgbm.readthedocs.io/en/v4.6.0/Parameters.html).
Weighting is for ranking under imbalance and does not establish calibrated risk probabilities.

In [3]:
def build_lightgbm(y_fit):
    weight = float(y_fit.eq(0).sum() / y_fit.eq(1).sum())
    return LGBMClassifier(**LGB_PARAMS, scale_pos_weight=weight)

display(pd.Series(LGB_PARAMS, name="fixed LightGBM settings").to_frame())
evaluate_configuration("E02", "LightGBM", False, build_lightgbm)
evaluate_configuration("E03", "LightGBM", True, build_lightgbm)

,fixed LightGBM settings
n_estimators,400
learning_rate,0.05
num_leaves,31
max_depth,5
min_child_samples,40
subsample,0.8
subsample_freq,1
colsample_bytree,1.0
reg_lambda,1.0
objective,binary


E02 | LightGBM | time=False | fold 1 AUC=0.921677
E02 | LightGBM | time=False | fold 2 AUC=0.994402
E02 | LightGBM | time=False | fold 3 AUC=0.998148
E03 | LightGBM | time=True | fold 1 AUC=0.931117
E03 | LightGBM | time=True | fold 2 AUC=0.994504
E03 | LightGBM | time=True | fold 3 AUC=0.998137


## 4. XGBoost experiments

E04 uses 165 features; E05 adds numeric `time_step`. Use CPU histogram trees with
fixed seed/thread count and the same fitting-only class-ratio policy. No early
stopping or search is applied to the outer validation folds. The fixed settings
follow the [XGBoost 3.0.5 parameter documentation](https://xgboost.readthedocs.io/en/release_3.0.0/parameter.html).

In [4]:
def build_xgboost(y_fit):
    weight = float(y_fit.eq(0).sum() / y_fit.eq(1).sum())
    return XGBClassifier(**XGB_PARAMS, scale_pos_weight=weight)

display(pd.Series(XGB_PARAMS, name="fixed XGBoost settings").to_frame())
evaluate_configuration("E04", "XGBoost", False, build_xgboost)
evaluate_configuration("E05", "XGBoost", True, build_xgboost)

,fixed XGBoost settings
n_estimators,400
learning_rate,0.05
max_depth,5
min_child_weight,5
subsample,0.8
colsample_bytree,1.0
reg_lambda,1.0
objective,binary:logistic
eval_metric,auc
tree_method,hist


E04 | XGBoost | time=False | fold 1 AUC=0.940656
E04 | XGBoost | time=False | fold 2 AUC=0.995593
E04 | XGBoost | time=False | fold 3 AUC=0.997495
E05 | XGBoost | time=True | fold 1 AUC=0.939144
E05 | XGBoost | time=True | fold 2 AUC=0.995304
E05 | XGBoost | time=True | fold 3 AUC=0.997513


## 5. Effect of adding time_step

Compare paired folds for each model family. Positive deltas favour including time.
This is a single-seed comparison, not proof of a general effect. Because future
validation times exceed all fitting times, ordinary tree thresholds cannot
extrapolate a new time trend beyond the fitted range. Time can still change the
partitions learned from past periods, so its actual effect must be measured.

In [5]:
fold_scores = pd.DataFrame(fold_results)
assert len(fold_scores) == 15
assert not fold_scores.duplicated(["experiment_id", "fold"]).any()
records = []
for experiment_id, scores in fold_scores.groupby("experiment_id", sort=False):
    scores = scores.sort_values("fold")
    aucs = scores["roc_auc"].to_numpy()
    records.append({"experiment_id": experiment_id, "model": scores.iloc[0]["model"],
                    "include_time_step": bool(scores.iloc[0]["include_time_step"]),
                    "n_features": int(scores.iloc[0]["n_features"]),
                    "fold_1_auc": float(aucs[0]), "fold_2_auc": float(aucs[1]), "fold_3_auc": float(aucs[2]),
                    "mean_auc": float(aucs.mean()), "std_auc": float(aucs.std(ddof=0)),
                    "latest_auc": float(aucs[2])})
comparison = pd.DataFrame(records)
time_deltas = []
for family in ["LightGBM", "XGBoost"]:
    family_rows = comparison.loc[comparison["model"].eq(family)].set_index("include_time_step")
    difference = family_rows.loc[True, ["fold_1_auc", "fold_2_auc", "fold_3_auc", "mean_auc", "std_auc"]] - family_rows.loc[False, ["fold_1_auc", "fold_2_auc", "fold_3_auc", "mean_auc", "std_auc"]]
    time_deltas.append({"model": family, **{f"delta_{key}": float(value) for key, value in difference.items()}})
time_deltas = pd.DataFrame(time_deltas)
display(time_deltas.round(6))
print("Deltas = with time minus without time; a lower standard deviation indicates less fold spread.")

,model,delta_fold_1_auc,delta_fold_2_auc,delta_fold_3_auc,delta_mean_auc,delta_std_auc
0,LightGBM,0.009440,0.000102,-0.000011,0.003177,-0.004426
1,XGBoost,-0.001512,-0.000289,0.000018,-0.000594,0.000653


Deltas = with time minus without time; a lower standard deviation indicates less fold spread.


## 6. Experiment comparison table

Report every fold, unweighted mean, population standard deviation, and latest-period
AUC. Compare the latest fold with E01's original 0.917141 and rolling results with
E01-R. Historical E01 public AUC is user-reported; new model public scores remain
unknown. All comparisons are kept within this notebook, with no separate log file.


In [6]:
display(comparison.round(6))
print("Original E01 local AUC:", E01_LOCAL_AUC,
      "| user-reported public Kaggle AUC:", E01_PUBLIC_AUC)
print("Public Kaggle scores for E02-E05 are unknown.")


,experiment_id,model,include_time_step,n_features,fold_1_auc,fold_2_auc,fold_3_auc,mean_auc,std_auc,latest_auc
0,E01-R,Logistic Regression,False,165,0.894508,0.933691,0.917141,0.915113,0.016060,0.917141
1,E02,LightGBM,False,165,0.921677,0.994402,0.998148,0.971409,0.035199,0.998148
2,E03,LightGBM,True,166,0.931117,0.994504,0.998137,0.974586,0.030773,0.998137
3,E04,XGBoost,False,165,0.940656,0.995593,0.997495,0.977915,0.026357,0.997495
4,E05,XGBoost,True,166,0.939144,0.995304,0.997513,0.977320,0.027010,0.997513


Original E01 local AUC: 0.917141264532569 | user-reported public Kaggle AUC: 0.88776
Public Kaggle scores for E02-E05 are unknown.


## 7. Final model selection

The rule was fixed before evaluation: (1) keep boosted configurations within
**0.002 AUC** of the best latest fold; (2) among those, keep configurations within
**0.001 AUC** of the best mean; (3) choose the lowest fold standard deviation, then
latest AUC and mean as tie-breakers. These are practical near-tie bands, not
confidence intervals. Latest-period quality takes priority without selecting solely
on a single peak. E01-R is a reference, not a boosted candidate.

These folds are also used for selection, so the winner's local score retains some
selection optimism. No prediction of leaderboard performance is made.

In [7]:
boosted = comparison.loc[comparison["model"].isin(["LightGBM", "XGBoost"])].copy()
latest_pool = boosted.loc[boosted["latest_auc"] >= boosted["latest_auc"].max() - LATEST_TOLERANCE]
mean_pool = latest_pool.loc[latest_pool["mean_auc"] >= latest_pool["mean_auc"].max() - MEAN_TOLERANCE]
ranked = mean_pool.sort_values(["std_auc", "latest_auc", "mean_auc", "experiment_id"],
                              ascending=[True, False, False, True])
selected = ranked.iloc[0]
selected_id = str(selected["experiment_id"])
selected_columns = ["time_step", *FEATURES] if selected["include_time_step"] else FEATURES
print("Latest-period eligible IDs:", latest_pool["experiment_id"].tolist())
print("Mean-AUC eligible IDs:", mean_pool["experiment_id"].tolist())
print("Selected:", selected_id, selected["model"], "| time_step:", bool(selected["include_time_step"]))
display(selected.to_frame("selected configuration"))
print("Latest-fold gain over E01:", float(selected["latest_auc"] - E01_LOCAL_AUC))

Latest-period eligible IDs: ['E02', 'E03', 'E04', 'E05']
Mean-AUC eligible IDs: ['E04', 'E05']
Selected: E04 XGBoost | time_step: False


,selected configuration
experiment_id,E04
model,XGBoost
include_time_step,False
n_features,165
fold_1_auc,0.940656
fold_2_auc,0.995593
fold_3_auc,0.997495
mean_auc,0.977915
std_auc,0.026357
latest_auc,0.997495


Latest-fold gain over E01: 0.08035395763656628


## 8. Training on all labelled data

Recreate the selected model with unchanged hyperparameters and recompute the
negative/positive weight from all labelled observations in steps 1–35. Test data
and unknown training labels are excluded from fitting. No additional model search
or post-selection retuning is performed.

In [8]:
assert labelled["time_step"].between(1, 35).all() and y.notna().all()
final_builder = build_lightgbm if selected["model"] == "LightGBM" else build_xgboost
final_model = final_builder(y)
with threadpool_limits(limits=1):
    final_model.fit(X[selected_columns], y)
assert final_model.classes_.tolist() == [0, 1]
assert final_model.n_features_in_ == len(selected_columns)
print("Final model fitted:", selected_id, "| rows:", len(y), "| features:", len(selected_columns),
      "| scale_pos_weight:", final_model.get_params()["scale_pos_weight"])

Final model fitted: E04 | rows: 31235 | features: 165 | scale_pos_weight: 7.571624588364434


## 9. Submission generation and integrity

Verify the embedded experiment identities before replacing the submission. These
fingerprints include model settings, features, folds, versions, and raw hashes.
Predict continuous class-1 probabilities and join to the template's literal `index`
key. Check columns, row count, exact key order, uniqueness, missingness, finiteness,
and probability bounds before saving and after reloading.

The only generated file is `outputs/submission_tabular_v2.csv`. Upload it to Kaggle
manually. Original E01 files and earlier experiment records are not needed here.


In [9]:
# Check the original experiment configurations before replacing a submission.
configurations = {}
for record in comparison.to_dict("records"):
    eid, family = record["experiment_id"], record["model"]
    params = LR_PARAMS if family == "Logistic Regression" else LGB_PARAMS if family == "LightGBM" else XGB_PARAMS
    columns = ["time_step", *FEATURES] if record["include_time_step"] else FEATURES
    config = {"model": family, "parameters": params, "features": columns, "folds": FOLDS,
              "weighting": "balanced in fitting labels" if family == "Logistic Regression" else "fitting negatives / fitting positives",
              "preprocessing": "fitting median imputation and standardization" if family == "Logistic Regression" else "infinity to NaN; native missing values",
              "seed": SEED, "numerical_threads": 1, "packages": VERSIONS, "raw_sha256": raw_hashes}
    config_id = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
    assert config_id == EXPECTED_CONFIG_IDS[eid], f"{eid} configuration changed; use a new experiment ID and update its recorded configuration."
    configurations[eid] = {"config_id": config_id, **config}

with threadpool_limits(limits=1):
    positive = int(np.flatnonzero(final_model.classes_ == 1)[0])
    test_probabilities = final_model.predict_proba(X_test[selected_columns])[:, positive]
predictions = pd.DataFrame({"index": test["index"].to_numpy(), "target": test_probabilities})
submission = sample[["index"]].merge(predictions, how="left", on="index", sort=False, validate="one_to_one")

def check_submission(frame):
    assert frame.columns.tolist() == ["index", "target"]
    assert len(frame) == len(test) == len(sample)
    assert frame["index"].equals(sample["index"]) and frame["index"].is_unique
    assert not frame.isna().any().any()
    assert np.isfinite(frame["target"]).all() and frame["target"].between(0, 1).all()

check_submission(submission)
for name, digest in raw_hashes.items():
    assert fingerprint(RAW / name) == digest, f"Raw file changed: {name}"
submission_path = OUT / "submission_tabular_v2.csv"
submission.to_csv(submission_path, index=False)
saved = pd.read_csv(submission_path)
check_submission(saved)
np.testing.assert_allclose(saved["target"], submission["target"], rtol=1e-6, atol=1e-12)

print("Saved outputs/submission_tabular_v2.csv:", len(saved), "rows; all checks passed.")
print("Selected model:", selected_id, "| configuration SHA-256:", configurations[selected_id]["config_id"])
print("Submission SHA-256:", fingerprint(submission_path))
print("Raw files unchanged. Upload this CSV to Kaggle manually.")


Saved outputs/submission_tabular_v2.csv: 15329 rows; all checks passed.
Selected model: E04 | configuration SHA-256: 739641a353d656a696e592db04371e7dc83a2ee0a760c61ea6e0123d76d30298
Submission SHA-256: 897bc0ac9708c1bb6ebbc52cd3bac6d0e84cc2948ced0351d49fec9416f2ec7f
Raw files unchanged. Upload this CSV to Kaggle manually.


## 10. Interpretation and limitations

Boosted trees improve all three folds relative to Logistic Regression, consistent
with nonlinear thresholds and feature interactions. These anonymous features do
not support an interpretation of specific transaction attributes. Time improves
LightGBM's mean and slightly worsens XGBoost's mean in this single-seed comparison;
its latest-fold effect is negligible for both models.

The selected E04 run achieved AUC 0.940656 / 0.995593 / 0.997495, mean 0.977915,
and population standard deviation 0.026357. Fold 1 is substantially harder. These
folds also select the winner, so there is some selection optimism. The near-perfect
latest-fold AUC is not a forecast of future Kaggle performance. Unknown feature
lineage and future-period distribution shift remain limitations. Weighted training
does not establish probability calibration.
